# Select and Download AIA Experiment Scenes

This notebook keeps data acquisition separate from the PSF deconvolution and file-size experiment.

It selects three independent anchors for each scene regime—quiet Sun, active region, and flare—and acquires one full-frame Level 1 image for each `aiapy` PSF-supported wavelength: 94, 131, 171, 193, 211, 304, and 335 Å.

## Selection logic

- **Quiet Sun:** solar-minimum candidates, retained after checking for significant flares and active-region context.
- **Active region:** three different NOAA active regions, sampled away from their listed major-flare peaks.
- **Flare:** three independent X-class events from different years and active regions, using reported peak times as anchors.

HEK event coverage and automated classifications are supporting evidence, not perfect ground truth. Keep validation notes with the manifest and manually inspect questionable scenes before freezing the labels.

AIA discovery uses one wavelength-independent ±7-second search per scene, then selects the nearest record for each supported wavelength. This accounts for the nominal 12-second cadence while limiting the final acquisition to exactly 63 files.

In [1]:
import re
from pathlib import Path

import astropy.units as u
import pandas as pd
import sunpy.map
from astropy.time import Time
from sunpy.net import Fido
from sunpy.net import attrs as a

DATA_ROOT = Path("data/aia_scene_experiment")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

WAVELENGTHS = (94, 131, 171, 193, 211, 304, 335)
VALIDATION_HALF_WINDOW = 12 * u.hour
IMAGE_HALF_WINDOW = 7 * u.second
EXPECTED_SCENES = 9
EXPECTED_PAIRS = EXPECTED_SCENES * len(WAVELENGTHS)

SCENES = [
    {
        "scene_id": "quiet_20190207",
        "scene_type": "quiet_sun",
        "anchor_time": "2019-02-07T12:00:00",
        "expected_event": "Solar-minimum quiet-Sun candidate",
        "selection_note": "Verify no NOAA active region and no M/X flare in the validation window.",
    },
    {
        "scene_id": "quiet_20190630",
        "scene_type": "quiet_sun",
        "anchor_time": "2019-06-30T12:00:00",
        "expected_event": "Solar-minimum quiet-Sun candidate",
        "selection_note": "Verify no NOAA active region and no M/X flare in the validation window.",
    },
    {
        "scene_id": "quiet_20191215",
        "scene_type": "quiet_sun",
        "anchor_time": "2019-12-15T12:00:00",
        "expected_event": "Solar-minimum quiet-Sun candidate",
        "selection_note": "Verify no NOAA active region and no M/X flare in the validation window.",
    },
    {
        "scene_id": "active_20120708_ar11520",
        "scene_type": "active_region",
        "anchor_time": "2012-07-08T12:00:00",
        "expected_event": "NOAA AR 11520",
        "selection_note": "Active-region morphology; confirm no M/X flare overlaps the image window.",
    },
    {
        "scene_id": "active_20141022_ar12192",
        "scene_type": "active_region",
        "anchor_time": "2014-10-22T12:00:00",
        "expected_event": "NOAA AR 12192",
        "selection_note": "Large active region; confirm the anchor is separated from major flare emission.",
    },
    {
        "scene_id": "active_20220329_ar12975_12976",
        "scene_type": "active_region",
        "anchor_time": "2022-03-29T12:00:00",
        "expected_event": "NOAA AR 12975/12976",
        "selection_note": "Active-region pair; confirm no M/X flare overlaps the image window.",
    },
    {
        "scene_id": "flare_20110215_x2p2",
        "scene_type": "flare",
        "anchor_time": "2011-02-15T01:56:00",
        "expected_event": "X2.2 flare, NOAA AR 11158",
        "selection_note": "Reported GOES peak; inspect saturation and diffraction structure.",
    },
    {
        "scene_id": "flare_20141024_x3p1",
        "scene_type": "flare",
        "anchor_time": "2014-10-24T21:41:00",
        "expected_event": "X3.1 flare, NOAA AR 12192",
        "selection_note": "Reported GOES peak; inspect saturation and diffraction structure.",
    },
    {
        "scene_id": "flare_20170910_x8p2",
        "scene_type": "flare",
        "anchor_time": "2017-09-10T16:06:00",
        "expected_event": "X8.2 flare, NOAA AR 12673",
        "selection_note": "Reported GOES peak; limb event with strong diffraction structure.",
    },
]

scene_manifest = pd.DataFrame(SCENES)
assert len(scene_manifest) == EXPECTED_SCENES
assert scene_manifest["scene_id"].is_unique
scene_manifest

,scene_id,scene_type,anchor_time,expected_event,selection_note
0,quiet_20190207,quiet_sun,2019-02-07T12:00:00,Solar-minimum quiet-Sun candidate,Verify no NOAA active region and no M/X flare ...
1,quiet_20190630,quiet_sun,2019-06-30T12:00:00,Solar-minimum quiet-Sun candidate,Verify no NOAA active region and no M/X flare ...
2,quiet_20191215,quiet_sun,2019-12-15T12:00:00,Solar-minimum quiet-Sun candidate,Verify no NOAA active region and no M/X flare ...
3,active_20120708_ar11520,active_region,2012-07-08T12:00:00,NOAA AR 11520,Active-region morphology; confirm no M/X flare...
4,active_20141022_ar12192,active_region,2014-10-22T12:00:00,NOAA AR 12192,Large active region; confirm the anchor is sep...
5,active_20220329_ar12975_12976,active_region,2022-03-29T12:00:00,NOAA AR 12975/12976,Active-region pair; confirm no M/X flare overl...
6,flare_20110215_x2p2,flare,2011-02-15T01:56:00,"X2.2 flare, NOAA AR 11158",Reported GOES peak; inspect saturation and dif...
7,flare_20141024_x3p1,flare,2014-10-24T21:41:00,"X3.1 flare, NOAA AR 12192",Reported GOES peak; inspect saturation and dif...
8,flare_20170910_x8p2,flare,2017-09-10T16:06:00,"X8.2 flare, NOAA AR 12673",Reported GOES peak; limb event with strong dif...


## Compile HEK flare context for all scene windows

A bare HEK flare query returns detections from every feature-recognition method. During active periods, one physical flare can have many duplicate automated detections, making those responses much slower than quiet-Sun responses.

The next cell queries only canonical `SWPC` reports for every scene and compiles two tables:

- `hek_window_summary`: one row per scene window, including class counts and an automatic scene-characteristic check.
- `hek_event_table`: one row per SWPC flare report across all windows, tagged with its scene and offset from the anchor.

The automatic rules are: quiet-Sun windows have no M/X flare in ±12 hours; active-region anchors have no M/X flare within ±30 minutes; flare anchors have an X-class report within ±30 minutes. These checks validate flare context only—active-region morphology and quiet-Sun appearance still require image inspection.

In [2]:
def response_to_dataframe(response) -> pd.DataFrame:
    """Combine all non-empty tables in a Fido response."""
    frames = [block.to_pandas() for block in response if len(block)]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def first_existing_column(frame: pd.DataFrame, names: tuple[str, ...]) -> str | None:
    lookup = {str(column).lower(): str(column) for column in frame.columns}
    return next(
        (lookup[name.lower()] for name in names if name.lower() in lookup), None
    )


def cleaned_unique_values(frame: pd.DataFrame, names: tuple[str, ...]) -> list[str]:
    column = first_existing_column(frame, names)
    if column is None:
        return []
    values = []
    for value in frame[column].dropna():
        text = str(value).strip()
        if text and text.lower() not in {"nan", "none", "0", "0.0"}:
            values.append(text)
    return sorted(set(values))


HEK_VALIDATION_HALF_WINDOW = 12 * u.hour
FLARE_MATCH_HALF_WINDOW = 30 * u.minute


def query_hek_flares(scene: pd.Series) -> tuple[pd.DataFrame, dict]:
    """Query canonical SWPC flare reports and summarize one scene window."""
    center = Time(scene["anchor_time"])
    start = center - HEK_VALIDATION_HALF_WINDOW
    end = center + HEK_VALIDATION_HALF_WINDOW
    flare_frame = response_to_dataframe(
        Fido.search(
            a.Time(start, end),
            a.hek.EventType("FL"),
            a.hek.FRM.Name == "SWPC",
        )
    )
    flare_frame = flare_frame.copy()
    flare_frame.insert(0, "scene_id", scene["scene_id"])
    flare_frame.insert(1, "scene_type", scene["scene_type"])
    flare_frame.insert(2, "anchor_time", scene["anchor_time"])

    class_column = first_existing_column(
        flare_frame, ("fl_goescls", "FL_GOESCls", "goes_class")
    )
    peak_column = first_existing_column(
        flare_frame, ("event_peaktime", "peak_time", "event_starttime", "start_time")
    )
    noaa_column = first_existing_column(
        flare_frame, ("ar_noaanum", "AR_NOAANum", "noaa_ar")
    )

    if class_column is None:
        flare_frame["goes_class_normalized"] = ""
    else:
        flare_frame["goes_class_normalized"] = (
            flare_frame[class_column].fillna("").astype(str).str.strip().str.upper()
        )
    flare_frame["is_m_or_x"] = flare_frame["goes_class_normalized"].str.startswith(
        ("M", "X")
    )
    flare_frame["is_x_class"] = flare_frame["goes_class_normalized"].str.startswith("X")

    center_timestamp = pd.Timestamp(scene["anchor_time"]).tz_localize("UTC")
    if peak_column is None:
        flare_frame["event_time_utc"] = pd.NaT
        flare_frame["anchor_offset_minutes"] = float("nan")
    else:
        flare_frame["event_time_utc"] = pd.to_datetime(
            flare_frame[peak_column], errors="coerce", utc=True
        )
        flare_frame["anchor_offset_minutes"] = (
            flare_frame["event_time_utc"] - center_timestamp
        ).dt.total_seconds() / 60
    flare_frame["near_anchor"] = flare_frame[
        "anchor_offset_minutes"
    ].abs() <= FLARE_MATCH_HALF_WINDOW.to_value(u.minute)

    major_context = flare_frame["is_m_or_x"]
    major_near = major_context & flare_frame["near_anchor"]
    x_near = flare_frame["is_x_class"] & flare_frame["near_anchor"]

    if scene["scene_type"] == "quiet_sun":
        characteristic_pass = not major_context.any()
        criterion = "No M/X SWPC flare in ±12 h"
    elif scene["scene_type"] == "active_region":
        characteristic_pass = not major_near.any()
        criterion = "No M/X SWPC flare within ±30 min"
    else:
        characteristic_pass = x_near.any()
        criterion = "At least one X-class SWPC flare within ±30 min"

    summary = {
        "scene_id": scene["scene_id"],
        "scene_type": scene["scene_type"],
        "anchor_time": scene["anchor_time"],
        "expected_event": scene["expected_event"],
        "validation_start": start.isot,
        "validation_end": end.isot,
        "swpc_flare_records": len(flare_frame),
        "c_class_records": int(
            flare_frame["goes_class_normalized"].str.startswith("C").sum()
        ),
        "m_class_records": int(
            flare_frame["goes_class_normalized"].str.startswith("M").sum()
        ),
        "x_class_records": int(flare_frame["is_x_class"].sum()),
        "major_near_anchor_records": int(major_near.sum()),
        "x_near_anchor_records": int(x_near.sum()),
        "goes_classes": ",".join(
            cleaned_unique_values(flare_frame, ("goes_class_normalized",))
        ),
        "noaa_active_regions_in_flare_reports": ",".join(
            cleaned_unique_values(flare_frame, (noaa_column,)) if noaa_column else []
        ),
        "automatic_criterion": criterion,
        "automatic_characteristic_pass": bool(characteristic_pass),
        "query_status": "ok",
    }
    return flare_frame, summary


window_summaries = []
event_frames = []

for _, scene in scene_manifest.iterrows():
    print(f"Validating scene: {scene['scene_id']}")
    try:
        scene_events, scene_summary = query_hek_flares(scene)
        window_summaries.append(scene_summary)
        event_frames.append(scene_events)
    except Exception as exc:
        window_summaries.append(
            {
                "scene_id": scene["scene_id"],
                "scene_type": scene["scene_type"],
                "anchor_time": scene["anchor_time"],
                "expected_event": scene["expected_event"],
                "automatic_characteristic_pass": False,
                "query_status": f"error: {type(exc).__name__}: {exc}",
            }
        )

hek_window_summary = pd.DataFrame(window_summaries)
hek_event_table = (
    pd.concat(event_frames, ignore_index=True) if event_frames else pd.DataFrame()
)

hek_window_summary.to_csv(DATA_ROOT / "hek_window_summary.csv", index=False)
hek_event_table.to_csv(DATA_ROOT / "hek_flare_events.csv", index=False)

event_display_columns = [
    column
    for column in (
        "scene_id",
        "scene_type",
        "anchor_time",
        "event_time_utc",
        "anchor_offset_minutes",
        "goes_class_normalized",
        "is_m_or_x",
        "is_x_class",
        "near_anchor",
        "ar_noaanum",
        "event_starttime",
        "event_peaktime",
        "event_endtime",
    )
    if column in hek_event_table.columns
]

display(hek_window_summary)
display(hek_event_table[event_display_columns])

Validating scene: quiet_20190207
Validating scene: quiet_20190630
Validating scene: quiet_20191215
Validating scene: active_20120708_ar11520
Validating scene: active_20141022_ar12192
Validating scene: active_20220329_ar12975_12976
Validating scene: flare_20110215_x2p2
Validating scene: flare_20141024_x3p1
Validating scene: flare_20170910_x8p2


,scene_id,scene_type,anchor_time,expected_event,validation_start,validation_end,swpc_flare_records,c_class_records,m_class_records,x_class_records,major_near_anchor_records,x_near_anchor_records,goes_classes,noaa_active_regions_in_flare_reports,automatic_criterion,automatic_characteristic_pass,query_status
0,quiet_20190207,quiet_sun,2019-02-07T12:00:00,Solar-minimum quiet-Sun candidate,2019-02-07T00:00:00.000,2019-02-08T00:00:00.000,0,0,0,0,0,0,,,No M/X SWPC flare in ±12 h,True,ok
1,quiet_20190630,quiet_sun,2019-06-30T12:00:00,Solar-minimum quiet-Sun candidate,2019-06-30T00:00:00.000,2019-07-01T00:00:00.000,0,0,0,0,0,0,,,No M/X SWPC flare in ±12 h,True,ok
2,quiet_20191215,quiet_sun,2019-12-15T12:00:00,Solar-minimum quiet-Sun candidate,2019-12-15T00:00:00.000,2019-12-16T00:00:00.000,0,0,0,0,0,0,,,No M/X SWPC flare in ±12 h,True,ok
3,active_20120708_ar11520,active_region,2012-07-08T12:00:00,NOAA AR 11520,2012-07-08T00:00:00.000,2012-07-09T00:00:00.000,17,13,4,0,1,0,"C3.6,C4.4,C5.4,C5.9,C6.1,C6.6,C6.7,C6.8,C6.9,C...","11513,11514,11515,11519,11520",No M/X SWPC flare within ±30 min,False,ok
4,active_20141022_ar12192,active_region,2014-10-22T12:00:00,NOAA AR 12192,2014-10-22T00:00:00.000,2014-10-23T00:00:00.000,8,4,3,1,0,0,"C3.2,C4.6,C5.7,C6.5,M1.4,M2.7,M8.7,X1.6",12192,No M/X SWPC flare within ±30 min,True,ok
5,active_20220329_ar12975_12976,active_region,2022-03-29T12:00:00,NOAA AR 12975/12976,2022-03-29T00:00:00.000,2022-03-30T00:00:00.000,21,17,4,0,0,0,"C1.8,C2.1,C2.2,C2.3,C3.1,C3.3,C3.4,C3.5,C3.9,C...","12975,12978",No M/X SWPC flare within ±30 min,True,ok
6,flare_20110215_x2p2,flare,2011-02-15T01:56:00,"X2.2 flare, NOAA AR 11158",2011-02-14T13:56:00.000,2011-02-15T13:56:00.000,10,7,1,1,1,1,"B7.9,C1.0,C1.2,C2.7,C4.8,C6.6,C7.0,M2.2,X2.2",11158,At least one X-class SWPC flare within ±30 min,True,ok
7,flare_20141024_x3p1,flare,2014-10-24T21:41:00,"X3.1 flare, NOAA AR 12192",2014-10-24T09:41:00.000,2014-10-25T09:41:00.000,5,4,0,1,1,1,"C3.6,C4.4,C5.1,C9.2,X3.1",12192,At least one X-class SWPC flare within ±30 min,True,ok
8,flare_20170910_x8p2,flare,2017-09-10T16:06:00,"X8.2 flare, NOAA AR 12673",2017-09-10T04:06:00.000,2017-09-11T04:06:00.000,4,3,0,1,1,1,"C1.0,C1.6,C2.9,X8.2",12673,At least one X-class SWPC flare within ±30 min,True,ok


,scene_id,scene_type,anchor_time,event_time_utc,anchor_offset_minutes,goes_class_normalized,is_m_or_x,is_x_class,near_anchor,ar_noaanum,event_starttime,event_peaktime,event_endtime
0,active_20120708_ar11520,active_region,2012-07-08T12:00:00,2012-07-08 00:09:00+00:00,-711.0,C8.4,False,False,False,11513.0,2012-07-07 23:56:00,2012-07-08 00:09:00,2012-07-08 00:16:00
1,active_20120708_ar11520,active_region,2012-07-08T12:00:00,2012-07-08 02:09:00+00:00,-591.0,C3.6,False,False,False,11515.0,2012-07-08 02:03:00,2012-07-08 02:09:00,2012-07-08 02:14:00
2,active_20120708_ar11520,active_region,2012-07-08T12:00:00,2012-07-08 03:32:00+00:00,-508.0,C4.4,False,False,False,11514.0,2012-07-08 03:28:00,2012-07-08 03:32:00,2012-07-08 03:35:00
3,active_20120708_ar11520,active_region,2012-07-08T12:00:00,2012-07-08 04:19:00+00:00,-461.0,C6.6,False,False,False,11515.0,2012-07-08 04:14:00,2012-07-08 04:19:00,2012-07-08 04:24:00
4,active_20120708_ar11520,active_region,2012-07-08T12:00:00,2012-07-08 05:34:00+00:00,-386.0,C5.9,False,False,False,11515.0,2012-07-08 05:28:00,2012-07-08 05:34:00,2012-07-08 05:40:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,flare_20141024_x3p1,flare,2014-10-24T21:41:00,2014-10-25 08:08:00+00:00,627.0,C9.2,False,False,False,12192.0,2014-10-25 07:36:00,2014-10-25 08:08:00,2014-10-25 08:35:00
61,flare_20170910_x8p2,flare,2017-09-10T16:06:00,2017-09-10 09:20:00+00:00,-406.0,C2.9,False,False,False,12673.0,2017-09-10 09:02:00,2017-09-10 09:20:00,2017-09-10 09:34:00
62,flare_20170910_x8p2,flare,2017-09-10T16:06:00,2017-09-10 14:23:00+00:00,-103.0,C1.6,False,False,False,12673.0,2017-09-10 14:19:00,2017-09-10 14:23:00,2017-09-10 14:25:00
63,flare_20170910_x8p2,flare,2017-09-10T16:06:00,2017-09-10 15:26:00+00:00,-40.0,C1.0,False,False,False,12673.0,2017-09-10 15:23:00,2017-09-10 15:26:00,2017-09-10 15:28:00


## Check image availability with nine short searches

Run one wavelength-independent Fido query per scene over only ±7 seconds. This uses nine provider requests rather than expanding the seven wavelength constraints into dozens of requests.

The 14-second interval covers at least one nominal 12-second cadence cycle. It can still contain two records at an interval boundary, so the code selects the record nearest to the scene anchor for each supported wavelength. Unsupported AIA channels returned by the broad query are ignored. Only the resulting 63 selected records are eligible for download.

In [3]:
def response_record_count(response) -> int:
    return sum(len(block) for block in response)


def wavelength_as_int(value) -> int:
    if hasattr(value, "to_value"):
        raw = value.to_value(u.angstrom)
        try:
            return int(round(float(raw)))
        except (TypeError, ValueError):
            return int(round(float(raw[0])))
    match = re.search(r"\d+(?:\.\d+)?", str(value))
    if match is None:
        raise ValueError(f"Cannot parse wavelength from {value!r}")
    return int(round(float(match.group())))


def search_aia_scene(scene: pd.Series):
    center = Time(scene["anchor_time"])
    start = center - IMAGE_HALF_WINDOW
    end = center + IMAGE_HALF_WINDOW
    return (
        Fido.search(
            a.Time(start, end),
            a.Instrument.aia,
            a.Level(1),
        ),
        start,
        end,
    )


def select_nearest_records(result, anchor_time: str):
    """Select exactly one nearest-in-time record for each supported wavelength."""
    center = Time(anchor_time)
    candidates = {wavelength: [] for wavelength in WAVELENGTHS}
    for block_index, block in enumerate(result):
        wavelength_column = next(
            (
                name
                for name in ("Wavelength", "wavelength", "wave")
                if name in block.colnames
            ),
            None,
        )
        start_column = next(
            (
                name
                for name in ("Start Time", "start_time", "time")
                if name in block.colnames
            ),
            None,
        )
        if wavelength_column is None or start_column is None:
            continue
        for row_index, row in enumerate(block):
            wavelength = wavelength_as_int(row[wavelength_column])
            if wavelength not in candidates:
                continue
            start_time = Time(row[start_column])
            offset_seconds = abs((start_time - center).to_value(u.second))
            candidates[wavelength].append(
                (offset_seconds, start_time.isot, block_index, row_index)
            )

    selected = {}
    summary = {}
    for wavelength, rows in candidates.items():
        rows.sort(key=lambda item: (item[0], item[1]))
        summary[wavelength] = rows
        if rows:
            _, _, block_index, row_index = rows[0]
            selected[wavelength] = result[block_index, row_index]
    return selected, summary


scene_queries = {}
selected_records = {}
availability_rows = []

for _, scene in scene_manifest.iterrows():
    try:
        result, start, end = search_aia_scene(scene)
        scene_queries[scene["scene_id"]] = result
        selected, candidates = select_nearest_records(result, scene["anchor_time"])
        for wavelength in WAVELENGTHS:
            key = (scene["scene_id"], wavelength)
            if wavelength in selected:
                selected_records[key] = selected[wavelength]
            rows = candidates[wavelength]
            availability_rows.append(
                {
                    "scene_id": scene["scene_id"],
                    "scene_type": scene["scene_type"],
                    "anchor_time": scene["anchor_time"],
                    "wavelength_angstrom": wavelength,
                    "search_start": start.isot,
                    "search_end": end.isot,
                    "candidate_records": len(rows),
                    "selected_records": int(wavelength in selected),
                    "selected_start_time": rows[0][1] if rows else None,
                    "selected_offset_seconds": rows[0][0] if rows else None,
                    "scene_candidate_records": response_record_count(result),
                    "response_blocks": len(result),
                    "search_status": "ok",
                }
            )
    except Exception as exc:
        for wavelength in WAVELENGTHS:
            availability_rows.append(
                {
                    "scene_id": scene["scene_id"],
                    "scene_type": scene["scene_type"],
                    "anchor_time": scene["anchor_time"],
                    "wavelength_angstrom": wavelength,
                    "candidate_records": 0,
                    "selected_records": 0,
                    "selected_start_time": None,
                    "selected_offset_seconds": None,
                    "scene_candidate_records": 0,
                    "response_blocks": 0,
                    "search_status": f"error: {type(exc).__name__}: {exc}",
                }
            )

availability = pd.DataFrame(availability_rows)
availability.to_csv(DATA_ROOT / "aia_availability.csv", index=False)
availability

,scene_id,scene_type,anchor_time,wavelength_angstrom,search_start,search_end,candidate_records,selected_records,selected_start_time,selected_offset_seconds,scene_candidate_records,response_blocks,search_status
0,quiet_20190207,quiet_sun,2019-02-07T12:00:00,94,2019-02-07T11:59:53.000,2019-02-07T12:00:07.000,1,1,2019-02-07T11:59:59.000,1.0,10,1,ok
1,quiet_20190207,quiet_sun,2019-02-07T12:00:00,131,2019-02-07T11:59:53.000,2019-02-07T12:00:07.000,2,1,2019-02-07T11:59:54.000,6.0,10,1,ok
2,quiet_20190207,quiet_sun,2019-02-07T12:00:00,171,2019-02-07T11:59:53.000,2019-02-07T12:00:07.000,1,1,2019-02-07T11:59:57.000,3.0,10,1,ok
3,quiet_20190207,quiet_sun,2019-02-07T12:00:00,193,2019-02-07T11:59:53.000,2019-02-07T12:00:07.000,1,1,2019-02-07T12:00:04.000,4.0,10,1,ok
4,quiet_20190207,quiet_sun,2019-02-07T12:00:00,211,2019-02-07T11:59:53.000,2019-02-07T12:00:07.000,1,1,2019-02-07T11:59:57.000,3.0,10,1,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,flare_20170910_x8p2,flare,2017-09-10T16:06:00,171,2017-09-10T16:05:53.000,2017-09-10T16:06:07.000,1,1,2017-09-10T16:05:57.000,3.0,9,1,ok
59,flare_20170910_x8p2,flare,2017-09-10T16:06:00,193,2017-09-10T16:05:53.000,2017-09-10T16:06:07.000,1,1,2017-09-10T16:06:07.000,7.0,9,1,ok
60,flare_20170910_x8p2,flare,2017-09-10T16:06:00,211,2017-09-10T16:05:53.000,2017-09-10T16:06:07.000,1,1,2017-09-10T16:05:57.000,3.0,9,1,ok
61,flare_20170910_x8p2,flare,2017-09-10T16:06:00,304,2017-09-10T16:05:53.000,2017-09-10T16:06:07.000,2,1,2017-09-10T16:06:05.000,5.0,9,1,ok


In [4]:
pair_duplicates = availability.duplicated(
    subset=["scene_id", "wavelength_angstrom"], keep=False
)
missing_pairs = availability.loc[availability["candidate_records"] == 0]
unselected_pairs = availability.loc[availability["selected_records"] != 1]

assert len(availability) == EXPECTED_PAIRS, (
    f"Expected {EXPECTED_PAIRS} search rows, found {len(availability)}."
)
assert not pair_duplicates.any(), "Duplicate scene–wavelength rows found."
assert missing_pairs.empty, f"Missing AIA candidates:\n{missing_pairs}"
assert unselected_pairs.empty, (
    "Every pair must have exactly one nearest-in-time selection. "
    f"Review these rows:\n{unselected_pairs}"
)
assert len(selected_records) == EXPECTED_PAIRS

acquisition_manifest = availability.merge(
    scene_manifest,
    on=["scene_id", "scene_type", "anchor_time"],
    how="left",
    validate="many_to_one",
)
acquisition_manifest.to_csv(DATA_ROOT / "scene_manifest.csv", index=False)

print(
    f"Selected {len(acquisition_manifest)} files from "
    f"{int(acquisition_manifest['candidate_records'].sum())} cadence-level candidates."
)
acquisition_manifest.groupby(["scene_type", "wavelength_angstrom"]).size().unstack()

Selected 63 files from 79 cadence-level candidates.


wavelength_angstrom,94,131,171,193,211,304,335
scene_type,,,,,,,
active_region,3,3,3,3,3,3,3
flare,3,3,3,3,3,3,3
quiet_sun,3,3,3,3,3,3,3


## Download the frozen acquisition set

The download is intentionally guarded. Run the fast validation cells first, inspect the availability table, then set `DOWNLOAD = True`.

Each of the nine batched scene queries downloads its seven files into one scene directory. Existing files are never overwritten or deleted. SunPy reads each local FITS header to assign it to the correct manifest wavelength.

In [5]:
DOWNLOAD = True

download_rows = []

if not DOWNLOAD:
    print(
        "Download is disabled. Review acquisition_manifest, then set "
        "DOWNLOAD = True and rerun this cell."
    )
else:
    for scene in scene_manifest.itertuples(index=False):
        target_dir = DATA_ROOT / scene.scene_type / scene.scene_id
        target_dir.mkdir(parents=True, exist_ok=True)
        existing_before = {
            path.resolve() for path in target_dir.glob("*.fits*") if path.is_file()
        }
        records = [
            selected_records[(scene.scene_id, wavelength)] for wavelength in WAVELENGTHS
        ]

        try:
            fetched = Fido.fetch(
                *records,
                path=target_dir / "{file}",
                site="NSO",
                overwrite=False,
            )
            fetch_errors = "; ".join(
                str(error) for error in getattr(fetched, "errors", [])
            )
        except Exception as exc:
            fetch_errors = f"{type(exc).__name__}: {exc}"

        local_files = sorted(
            path.resolve() for path in target_dir.glob("*.fits*") if path.is_file()
        )
        for path in local_files:
            try:
                smap = sunpy.map.Map(path)
                wavelength = int(round(smap.wavelength.to_value(u.angstrom)))
                if wavelength not in WAVELENGTHS:
                    continue
                status = "existing" if path in existing_before else "downloaded"
                map_error = ""
            except Exception as exc:
                wavelength = None
                status = "invalid_fits"
                map_error = f"{type(exc).__name__}: {exc}"
            download_rows.append(
                {
                    "scene_id": scene.scene_id,
                    "scene_type": scene.scene_type,
                    "anchor_time": scene.anchor_time,
                    "wavelength_angstrom": wavelength,
                    "local_path": str(path),
                    "bytes": path.stat().st_size,
                    "status": status,
                    "errors": "; ".join(
                        text for text in (fetch_errors, map_error) if text
                    ),
                }
            )

    download_manifest = pd.DataFrame(download_rows)
    download_manifest.to_csv(DATA_ROOT / "download_manifest.csv", index=False)
    display(download_manifest)

Files Downloaded:   0%|          | 0/7 [00:00<?, ?file/s]

aia.lev1.94A_2019_02_07T11_59_59.12Z.image_lev1.fits:   0%|          | 0.00/10.5M [00:00<?, ?B/s]

aia.lev1.211A_2019_02_07T11_59_57.62Z.image_lev1.fits:   0%|          | 0.00/9.07M [00:00<?, ?B/s]

aia.lev1.131A_2019_02_07T11_59_54.62Z.image_lev1.fits:   0%|          | 0.00/8.11M [00:00<?, ?B/s]

aia.lev1.193A_2019_02_07T12_00_04.84Z.image_lev1.fits:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

...

aia.lev1.193A_2017_09_10T16_06_07.53Z.image_lev1.fits:   0%|          | 0.00/6.99M [00:00<?, ?B/s]

aia.lev1.211A_2017_09_10T16_05_57.63Z.image_lev1.fits:   0%|          | 0.00/9.86M [00:00<?, ?B/s]

aia.lev1.304A_2017_09_10T16_06_05.13Z.image_lev1.fits:   0%|          | 0.00/7.31M [00:00<?, ?B/s]

aia.lev1.335A_2017_09_10T16_06_00.63Z.image_lev1.fits:   0%|          | 0.00/7.17M [00:00<?, ?B/s]

,scene_id,scene_type,anchor_time,wavelength_angstrom,local_path,bytes,status,errors
0,quiet_20190207,quiet_sun,2019-02-07T12:00:00,131,C:\__SANDBOX__\SDAC\data\aia_scene_experiment\...,8107200,downloaded,
1,quiet_20190207,quiet_sun,2019-02-07T12:00:00,171,C:\__SANDBOX__\SDAC\data\aia_scene_experiment\...,11695680,downloaded,
2,quiet_20190207,quiet_sun,2019-02-07T12:00:00,193,C:\__SANDBOX__\SDAC\data\aia_scene_experiment\...,10690560,downloaded,
3,quiet_20190207,quiet_sun,2019-02-07T12:00:00,211,C:\__SANDBOX__\SDAC\data\aia_scene_experiment\...,9074880,downloaded,
4,quiet_20190207,quiet_sun,2019-02-07T12:00:00,304,C:\__SANDBOX__\SDAC\data\aia_scene_experiment\...,7174080,downloaded,
...,...,...,...,...,...,...,...,...
58,flare_20170910_x8p2,flare,2017-09-10T16:06:00,193,C:\__SANDBOX__\SDAC\data\aia_scene_experiment\...,6992640,downloaded,
59,flare_20170910_x8p2,flare,2017-09-10T16:06:00,211,C:\__SANDBOX__\SDAC\data\aia_scene_experiment\...,9864000,downloaded,
60,flare_20170910_x8p2,flare,2017-09-10T16:06:00,304,C:\__SANDBOX__\SDAC\data\aia_scene_experiment\...,7312320,downloaded,
61,flare_20170910_x8p2,flare,2017-09-10T16:06:00,335,C:\__SANDBOX__\SDAC\data\aia_scene_experiment\...,7171200,downloaded,


## Verify downloaded FITS products

After downloading, verify that there is exactly one local file per scene–wavelength pair, each file exists, SunPy recognizes a 4096×4096 map, the wavelength matches the manifest, and Level 1 metadata is present when exposed by the provider.

In [6]:
manifest_path = DATA_ROOT / "download_manifest.csv"
if not manifest_path.exists():
    raise RuntimeError("Run the guarded download cell with DOWNLOAD = True first.")

download_manifest = pd.read_csv(manifest_path)
pair_counts = download_manifest.groupby(["scene_id", "wavelength_angstrom"]).size()
assert len(pair_counts) == EXPECTED_PAIRS
assert (pair_counts == 1).all(), "Expected exactly one file for every pair."

verification_rows = []
for row in download_manifest.itertuples(index=False):
    path = Path(row.local_path)
    record = {
        "scene_id": row.scene_id,
        "scene_type": row.scene_type,
        "wavelength_angstrom": int(row.wavelength_angstrom),
        "local_path": str(path),
        "exists": path.is_file(),
        "shape_ok": False,
        "wavelength_ok": False,
        "level_metadata": None,
        "level_ok": False,
        "map_status": "not_checked",
    }
    if path.is_file():
        try:
            smap = sunpy.map.Map(path)
            actual_wavelength = int(round(smap.wavelength.to_value(u.angstrom)))
            level_value = smap.meta.get(
                "lvl_num", smap.meta.get("level", smap.meta.get("data_lev"))
            )
            record.update(
                {
                    "shape_ok": tuple(smap.data.shape) == (4096, 4096),
                    "wavelength_ok": actual_wavelength == int(row.wavelength_angstrom),
                    "level_metadata": level_value,
                    "level_ok": level_value is None or str(level_value).startswith("1"),
                    "map_status": "ok",
                }
            )
        except Exception as exc:
            record["map_status"] = f"error: {type(exc).__name__}: {exc}"
    verification_rows.append(record)

verification = pd.DataFrame(verification_rows)
verification.to_csv(DATA_ROOT / "fits_verification.csv", index=False)

assert verification["exists"].all()
assert verification["shape_ok"].all()
assert verification["wavelength_ok"].all()
assert verification["level_ok"].all()
assert (verification["map_status"] == "ok").all()

print(f"Verified all {len(verification)} downloaded AIA Level 1 scene files.")
verification.groupby(["scene_type", "wavelength_angstrom"]).size().unstack()

Verified all 63 downloaded AIA Level 1 scene files.


wavelength_angstrom,94,131,171,193,211,304,335
scene_type,,,,,,,
active_region,3,3,3,3,3,3,3
flare,3,3,3,3,3,3,3
quiet_sun,3,3,3,3,3,3,3


- **`hek_window_summary.csv`**: One row per scene, summarizing SWPC HEK flare context in the ±12-hour validation window (flare-class counts, near-anchor checks, and automatic pass/fail criterion).
- **`hek_flare_events.csv`**: One row per SWPC flare report found in the scene windows, with event timing, GOES class normalization, scene tags, and offset from anchor time.
- **`aia_availability.csv`**: One row per scene–wavelength pair, showing short-window AIA search results, candidate counts, nearest selected record timing, and search status.
- **`scene_manifest.csv`**: Final acquisition manifest for all selected scene–wavelength pairs, combining availability selection fields with scene metadata and notes.
- **`download_manifest.csv`**: One row per local FITS file discovered after fetch, including scene/wavelength assignment, local path, file size, download/existing status, and any fetch/map errors.
- **`fits_verification.csv`**: One row per downloaded file with integrity checks (file exists, 4096×4096 shape, wavelength match, Level 1 metadata check, and map-read status).
